# xgap: demo replay (step 2)

Thin by design -- no logic lives here. Mount Drive, clone/pull the repo,
run `setup_colab.sh`, call `scripts/run_demo_replay.py`. All control flow
lives in `xgap_code/` and `scripts/`; this notebook only sequences calls to
it.

Repo: https://github.com/AITEAM444/xgap (public)

**Run cell 1 before importing anything else, in every fresh runtime.** Colab's
`!` shell subprocess env does not propagate into this kernel's own Python
process, so `setup_colab.sh` cannot set `MUJOCO_GL` for you here -- see that
script's own comments.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

XGAP_DRIVE_ROOT = "/content/drive/MyDrive/xgap"
XGAP_REPO_URL = "https://github.com/AITEAM444/xgap.git"

## Get the code

Clones into Drive on first run, `git pull`s on every run after -- code/config
history stays on GitHub (the source of truth), Drive is just where it lives so
`setup_colab.sh` / scripts can read it and outputs can be written next to it.
No manual re-uploading of files to Drive after this point; push to GitHub and
re-run this cell instead.

In [ ]:
os.makedirs(XGAP_DRIVE_ROOT, exist_ok=True)

if os.path.isdir(f"{XGAP_DRIVE_ROOT}/.git"):
    # Already a clone -- re-point + fast-forward rather than a bare `git pull`.
    !git -C {XGAP_DRIVE_ROOT} remote set-url origin {XGAP_REPO_URL}
    !git -C {XGAP_DRIVE_ROOT} fetch origin
    !git -C {XGAP_DRIVE_ROOT} checkout -B master origin/master
else:
    # No `.git` here, but `git clone` refuses to clone into a non-empty directory --
    # and this directory is never actually empty in practice, since setup_colab.sh
    # points HF_HOME at a `.hf_cache/` subfolder of it. `git init` + `fetch` + `checkout`
    # only touches files git itself tracks, so it works in-place regardless of what
    # other untracked stuff (like `.hf_cache/`) already lives here.
    !git -C {XGAP_DRIVE_ROOT} init
    !git -C {XGAP_DRIVE_ROOT} remote add origin {XGAP_REPO_URL}
    !git -C {XGAP_DRIVE_ROOT} fetch origin
    !git -C {XGAP_DRIVE_ROOT} checkout -B master origin/master

## Environment rebuild

Idempotent -- safe to re-run. This also appends `nproc` / `nvidia-smi` /
`free -g` / installed library versions to `logs/env_meta.log` (Colab hardware
varies session to session), which satisfies the "log hardware in the first
cell" requirement without duplicating that logic here.

If this prints a restart banner, use *Runtime > Restart session* and re-run
this cell once (it will no-op on the already-satisfied install step) before
continuing.

In [ ]:
!bash {XGAP_DRIVE_ROOT}/setup_colab.sh

## Adopt resolved env vars into this kernel

`setup_colab.sh`'s own `export`s (`HF_HOME`, `HF_LEROBOT_HOME`,
`LIBERO_CONFIG_PATH`) only apply to ITS OWN subprocesses -- they vanish once
that script exits, same as any other `!`-cell `export`. Without this cell,
the next cell (a separate `!python ...` subprocess) sees none of them: caches
silently fall back to their unconfigured defaults, and `libero`'s interactive
dataset-path prompt reappears even though it already passed inside
`setup_colab.sh`. Reads `/content/.xgap_env` (written by the script above) into
`os.environ` in the KERNEL process instead, which every `!` cell for the rest
of this session DOES inherit -- same mechanism as why cell 1's `MUJOCO_GL`
works.

In [ ]:
with open("/content/.xgap_env") as f:
    for line in f:
        key, _, value = line.strip().partition("=")
        if key:
            os.environ[key] = value

for _k in ["XGAP_DRIVE_ROOT", "MUJOCO_GL", "HF_HOME", "HF_LEROBOT_HOME", "LIBERO_CONFIG_PATH"]:
    print(f"{_k}={os.environ.get(_k)}")

## Smoke run first

`configs/demo_replay_smoke.yaml`: 1 suite, 1 task, up to 5 episodes, both
`control_mode`s, with per-episode `.mp4` + trajectory plots saved locally
(`outputs/demo_replay_smoke/_local/videos/`) for visual inspection. See the
repo README, "How to read the first real run", for how to tell a crash apart
from an actual low-success-rate finding.

If re-running after a code change to the replay/logging/video pipeline
itself (not just a config change), clear old results first -- resume only
checks "does this episode's result file already exist", not whether it has
the fields/videos the current code would produce:

```python
!rm -rf /content/outputs/demo_replay_smoke
!rm -rf /content/drive/MyDrive/xgap/outputs/demo_replay_smoke
```

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay_smoke.yaml

## control_freq comparison (done -- kept for reference)

`configs/demo_replay_smoke_cf20.yaml` -- same task/episode/control_modes as
the smoke config above, `control_freq=20` instead of `10`. This was run:
`control_freq=20` is now confirmed correct (see README "control_freq was
wrong from the start" -- LIBERO's own source shows demos are collected and
replayed 1:1 at 20Hz; `demo_replay_smoke.yaml`'s default was corrected to
match) and gets the eef position trajectory to track the recorded demo
almost exactly. But grasping still fails at `control_freq=20` too --
`gripper_qpos` closes fully (nothing between the fingers) both times, where
the recorded demo shows a partial close (something between the fingers)
both times. So this cell is no longer diagnostic on its own -- kept so the
comparison is reproducible -- and the investigation moved to the two cells
below instead.</cell id="b92b2689">


In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay_smoke_cf20.yaml

## Sanity check against a KNOWN answer -- DONE, infrastructure confirmed sound

None of this project's own code runs in this cell -- just `lerobot`'s own
`lerobot-eval` CLI against `lerobot/pi05_libero_finetuned`, a checkpoint with
a published official number: **96% on LIBERO-10** (see
`docs/source/libero.mdx` in lerobot, "Reproducing published results"). No
50GB dataset download needed either -- eval only touches the environment and
the (small) policy checkpoint.

**Result:** every task that finished before the run was stopped hit
`running_success_rate=100.0%` on 5/5 episodes (20/20 episodes across 4
completed tasks). See README "Sanity check result: infrastructure is
confirmed sound, the bug is ours" for the full story, including two false
alarms along the way worth knowing about before reading this cell's output:
`Stepping through eval batches: N/5` is a *per-task* bar that resets to a
fresh 0% every time a new task starts (looked like an ongoing failure mid-run,
was actually just a fresh counter), and the piped-to-file tqdm log is huge
(one line per step) so skimming only the tail can land inside an
in-progress episode that structurally shows 0.0% until it finishes --
grep the file for `100%` to find actual per-task completions instead.

**Fixed below:** `--env.task_ids=[0]` now restricts to a single task, since
`--env.task=libero_10` alone evaluates all 10 tasks in the suite (5 episodes
*per task*, not 5 total as originally intended) -- this is what actually
made the first run take ~50 episodes instead of the quick 5-episode check
this cell was meant to be.

**Needs a GPU runtime** (Runtime > Change runtime type > GPU) -- CPU-only
inference of a VLA policy is slow enough to look hung.

**Gated dependency:** this checkpoint's VLM backbone needs
`google/paligemma-3b-pt-224`, which Google gates. One-time setup: accept
the license at https://huggingface.co/google/paligemma-3b-pt-224 while
logged into your HF account, then in a cell: `from huggingface_hub import
login; login(token="...")` (token from
https://huggingface.co/settings/tokens).

**Output redirected to a log file, not printed live** -- `lerobot-eval`'s
per-step logging is verbose enough that letting Colab render it live can
overload the browser tab and disconnect the runtime (hit this once). The
cell below writes everything to `/content/lerobot_eval.log` and only prints
the last 80 lines; `!cat /content/lerobot_eval.log` in a scratch cell if you
need the full log, and `grep -c '100%' /content/lerobot_eval.log` to count
completed-task markers without reading the whole thing.</cell id="bb029484">


In [ ]:
!lerobot-eval \
    --policy.path=lerobot/pi05_libero_finetuned \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval.log 2>&1
!tail -n 80 /content/lerobot_eval.log

## SmolVLA directly via `lerobot-eval` -- no xgap code

Same idea as the sanity-check cell above, but for the actual checkpoint
under test (`HuggingFaceVLA/smolvla_libero`) instead of the known-good
`pi05` reference. This is the direct answer key for `xgap`'s own N=1 result
-- and doubles as the Gate-1 baseline -- without needing the demo-replay
init_state mapping at all (see README "Mapping search abandoned": policy
eval never uses a demo's initial state, only `init_states` in plain order,
which is exactly what this runs).

No `--policy.n_action_steps` override -- let the checkpoint's own config
supply it rather than reusing `pi05`'s `10` by copy-paste. Same GPU
requirement and log-redirect reasoning as the cell above.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla.log

**Result: 0/5, confirmed via pure `lerobot-eval` -- checkpoint-specific, not
an xgap harness bug.** See README "Mapping search abandoned" section for the
full writeup. Wrist-camera input (H2) is not a live suspect for this result
(`lerobot-eval`'s own env supplies both cameras automatically). Next: H1 --
sweep `n_action_steps` one value at a time, still via pure `lerobot-eval`.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla_nas10.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla_nas10.log

**Result: 0/5 again -- H1 (execution granularity) rejected, 1 vs 10 makes
no difference.** `eval_ep_s` dropped 462s -> 145.6s (~3.2x) on a GPU
upgrade (T4->A100) *and* 10x fewer policy calls combined -- much less than
compute-bound scaling would predict, so the real bottleneck is very likely
env stepping/rendering, not policy inference (see README).

Two higher-information, untested axes come next instead of continuing this
sweep to 25/50 (which would only add a third "0/5, learned nothing new"
point) -- see README "Higher-priority than finishing the `n_action_steps`
sweep": (1) does this checkpoint know `libero_10` at all (try
`libero_spatial` instead), (2) the unresolved 360-vs-256 resolution
question from Step 1.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_spatial \
    --env.task_ids=[0] \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla_spatial_nas10.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla_spatial_nas10.log

**Result: 3/5 (60%) -- decisive.** Checkpoint/policy/env/`lerobot-eval` loop
all genuinely work; the failure is specific to `libero_10`, not SmolVLA in
general. Also weakens (doesn't kill) the resolution hypothesis below: this
ran at the same default 360 resolution as every failing `libero_10` run and
still succeeded 60% of the time. See README for the full writeup and the
two remaining candidate explanations (task difficulty vs something
`libero_10`-scene-specific).

## Resolution mismatch check (unresolved since Step 1)

Env default render is 360, checkpoint declares 256, and there's a separate
512-padding setting on top -- never confirmed from source whether/where a
resize actually reconciles these. Check the real CLI field names first
(cheap, no GPU) so the eval below doesn't run on a typo'd flag:

In [ ]:
import dataclasses
from lerobot.envs.configs import LiberoEnv
print([f.name for f in dataclasses.fields(LiberoEnv)])

**Confirmed:** `observation_height`/`observation_width` are real fields
(full list: `task`, `fps`, `features`, `features_map`, `max_parallel_tasks`,
`disable_env_checker`, `task_ids`, `episode_length`, `obs_type`,
`render_mode`, `camera_name`, `init_states`, `camera_name_mapping`,
`observation_height`, `observation_width`, `is_libero_plus`,
`control_mode`) -- the eval cell below uses the right flag names.

In [ ]:
!lerobot-eval \
    --policy.path=HuggingFaceVLA/smolvla_libero \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --env.task_ids=[0] \
    --env.observation_height=256 \
    --env.observation_width=256 \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1 \
    > /content/lerobot_eval_smolvla_res256.log 2>&1
!tail -n 80 /content/lerobot_eval_smolvla_res256.log

## Gripper-close duration, not just pass/fail

Three 0/5s in a row say nothing new on their own -- but whether the longest
continuous closed-gripper run (`xgap_code/gripper_metrics.longest_close_run`;
a real demo grasp holds closed 15-20+ consecutive steps, measured earlier
from real demo data) moves between conditions splits "never attempts to
close" from "closes but doesn't grasp" -- different bugs. Check whether
plain `lerobot-eval` already saves raw per-step actions anywhere before
assuming a new script is needed:

In [ ]:
!ls -la outputs/eval/*/*/
!python -c "import lerobot.scripts.eval as e; print(e.__file__)"

## init_state sweep (is `within_task_index` picking the wrong index?)

`configs/sweep_init_states.yaml`: fixes ONE demo's recorded actions
(`libero_10` task 0, dataset episode_index=8 -- the same episode used in the
comparisons above) and replays them against **every** candidate `init_state`
LIBERO has for this task (`scripts/sweep_init_states.py`, no new
instrumentation -- one loop over `harness.get_num_init_states()`). Cheap
relative to another position/orientation investigation: same ~300-step
episode run N times (N = however many init_states this task has, LIBERO's
own count, not assumed), no additional download.

- **Any index succeeds** -> `within_task_index` (`dataset_io.py`)
  is picking the wrong LIBERO init_state, confirmed, and this sweep hands
  you the *correct* index directly (`sweep_summary.json`'s
  `successful_init_states`).
- **None succeed** -> init-state indexing is exonerated. Orientation
  (`eef_quat`, not currently in `state_chunk`) becomes the next thing to
  add and check -- see README.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/sweep_init_states.py \
    --config {XGAP_DRIVE_ROOT}/configs/sweep_init_states.yaml

## Visually verify one successful init_state

`scripts/sweep_init_states.py`'s resume logic keys on "does this episode's
result already exist" -- turning video on for an index it already ran would
just skip it silently and never produce a video. This is a separate,
always-re-runs script for spot-checking any ONE index from the sweep's
`successful_init_states` with an actual `.mp4` + trajectory-vs-demo overlay
plot, without touching the sweep's own results. Swap `--init-state-index`
for whichever index you want to look at.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/render_init_state_video.py \
    --config {XGAP_DRIVE_ROOT}/configs/sweep_init_states.yaml \
    --init-state-index 3 \
    --out-dir /content/outputs/init_state_render

## Full demo replay

Only run this after the smoke cell above completes cleanly (prints a
`decision` JSON, not a traceback).

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay.yaml